In [ ]:
import os
import glob
from pathlib import Path
from pathlib import PurePosixPath

In [ ]:
# Wenn Permission denied Fehlermeldung im Terminal
#chmod +x flair_vectorise.sh

In [ ]:
# Pfad zum Hauptverzeichnis
dir_input = '/mnt/eo/projekt/2023_Essnet/results/Raster_tiled/'
#dir_output = '/mnt/eo/projekt/2023_Essnet/results/Vektor_clipped/'
yamlfile_out_dir = '/mnt/eo/projekt/2023_Essnet/results/yamlfiles/'


In [ ]:
def update_yaml(yamlfile_out, input_raster):
    def update_yaml_text(key, new_value, file_yaml_input, file_yaml_output):
        # Ensure string values are quoted
        if isinstance(new_value, str) and not (new_value.startswith('"') or new_value.startswith("'")):
            new_value = f'"{new_value}"'
            print(new_value)

        #Read template file and replace values
        lines_out = []
        with open(file_yaml_input, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip().startswith(f"{key}:"):
                    lines_out.append(f'{key}: {new_value}\n')
                else:
                    lines_out.append(line)
    

        # # Write updated file
        with open(file_yaml_output, "w", encoding="utf-8") as f:
            f.writelines(lines_out)


    filename = yamlfile_out.split('/')
    filename = filename[-1].removesuffix('.yaml')
    
    # File Output !!!
    outputname_geopackage = '/home/nnors/Vektor/' + filename + '.gpkg'
    
    try:
        # Works in .py scripts
        file_yaml_proto = Path(__file__).resolve().parent
    except NameError:
        # Fallback for notebooks
        file_yaml_proto = Path.cwd()

    # is adding 'data/' infront of the path which is not there
    file_yaml_proto = Path(file_yaml_proto)
    file_yaml_proto = Path(str(file_yaml_proto).removeprefix('/data'))
    

    # name of proto yaml file
    file_yaml_proto = f'{file_yaml_proto}/config_vector_proto.yaml'



    # Update specific fields
    
    update_yaml_text(key = "input_file", new_value = input_raster, 
                     file_yaml_input = file_yaml_proto, file_yaml_output = yamlfile_out)
    
    update_yaml_text(key = "output_file", new_value = outputname_geopackage,
                     file_yaml_input = yamlfile_out, file_yaml_output = yamlfile_out)


In [ ]:
def list_tif_files(directory):
    return glob.glob(os.path.join(dir_input, "**", "*.tif"), recursive=True)

In [ ]:
list_input_rasters = list_tif_files(dir_input)

In [ ]:
for input_raster in list_input_rasters:
    input_filepath = input_raster.removesuffix('.tif')


    yamlfile_name = input_filepath.split('/')
    
    yamlfile_name = yamlfile_name[-1]


    yamlfile_out = f'{yamlfile_out_dir}{yamlfile_name}.yaml'

   

    
    # create and update yaml files
    update_yaml(yamlfile_out, input_raster)
    
    # Leerzeichen ist wichtig
    osCommand="/home/nnors/Documents/Essnet/FLAIR_1_fork/FLAIR-1/src/flair_aiml4os_vector_conv/flair_vectorise.sh " + yamlfile_out
    print(osCommand)

    if os.system(osCommand)!=0:
         print("error")
         exit(1)


